In [ ]:
"""
실시간 심박수 + 호흡수 측정 + 환자별 개인화 이상 탐지

  심박수  : open-rppg (rPPG)
  호흡수  : EVM Motion (라플라시안 피라미드 + 시간축 밴드패스 + PC1)
  이상탐지: Isolation Forest (환자별 기준 분포 학습)  → vital_anomaly.py

설치:
    pip install open-rppg opencv-python numpy scipy scikit-learn

실행:
    python vital_monitor.py --camera 0
    (Jupyter 에서는 rppg 의 스레드 정리와 충돌하므로 터미널에서 실행할 것)
"""

import argparse
import time
from collections import deque

import cv2
import numpy as np
from scipy.signal import butter, filtfilt

import rppg

from vital_anomaly import VitalAnomalyDetector


# ══════════════════════════════════════════════════════
#  스펙트럼 유틸
# ══════════════════════════════════════════════════════

def spectral_peak(sig, fs, f_low, f_high):
    """
    1D 시계열 → (BPM, SQI 0~1)

    SQI = 대역 내 피크 주변 파워 / 대역 전체 파워
          파워비이므로 창 길이에 불변. 백색잡음일 때 0 이 되도록 정규화.
    """
    n = len(sig)
    if n < 32:
        return 0.0, 0.0

    win = (sig - sig.mean()) * np.hanning(n)
    power = np.abs(np.fft.rfft(win, n=n * 4)) ** 2
    freqs = np.fft.rfftfreq(n * 4, d=1.0 / fs)

    band = (freqs >= f_low) & (freqs <= f_high)
    if not np.any(band):
        return 0.0, 0.0
    bp, bf = power[band], freqs[band]

    f0 = bf[int(np.argmax(bp))]

    sig_bins = np.abs(bf - f0) <= 0.03
    if 2 * f0 <= f_high:
        # 실제 호흡은 들숨/날숨이 비대칭이어서 파워가 f0 와 2f0 에 나뉜다.
        # 1차 고조파를 신호로 인정하지 않으면 좋은 신호도 SQI 가 깎인다.
        sig_bins = sig_bins | (np.abs(bf - 2 * f0) <= 0.03)
    ratio = float(bp[sig_bins].sum() / (bp.sum() + 1e-12))
    floor = sig_bins.sum() / len(bf)          # 백색잡음일 때의 기대 비율
    sqi = float(np.clip((ratio - floor) / (1.0 - floor), 0.0, 1.0))

    return float(f0 * 60.0), sqi


def laplacian_level(img, level):
    """가우시안 피라미드 level 단계의 라플라시안 성분 (2D float32)."""
    g = img
    for _ in range(level):
        g = cv2.pyrDown(g)
    return g - cv2.pyrUp(cv2.pyrDown(g))


# ══════════════════════════════════════════════════════
#  EVM 호흡수
# ══════════════════════════════════════════════════════

class RespiratorySignal:
    """
    EVM Motion 기반 호흡수 추정기.

    프레임마다 ROI 를 8x8 라플라시안 계수로 축약해 (시각, 벡터) 로 버퍼링하고,
    estimate() 호출 시에만 리샘플링 → 밴드패스 → PC1 → FFT 를 수행한다.
    프레임당 비용이 O(1) 이라 라즈베리파이에서도 부담이 적다.
    """

    FREQ_LOW, FREQ_HIGH = 0.15, 0.60   # Hz → 9 ~ 36 BPM
    TARGET_FS = 5.0                    # 리샘플 목표 (Nyquist 2.5Hz, 여유 4배)
    WINDOW_SEC = 60.0                  # 분석 창 (0.15Hz 기준 9주기)
    ROI_SIZE = (64, 64)
    PYR_LEVEL = 3                      # 64 → 8x8 (호흡은 큰 스케일 움직임)
    MIN_FILL = 0.5                     # 창의 이 비율은 차야 추정 시도

    def __init__(self):
        self.buf = deque()             # (t, flat_vec)
        self.rr_bpm = 0.0
        self.rr_conf = 0.0
        nyq = self.TARGET_FS / 2.0
        self.b, self.a = butter(2, [self.FREQ_LOW / nyq,
                                    self.FREQ_HIGH / nyq], btype="band")

    def clear(self):
        """ROI 종류가 바뀌면 호출 (신호 불연속 방지)."""
        self.buf.clear()
        self.rr_conf = 0.0

    def push(self, roi_bgr, t):
        gray = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
        gray = cv2.resize(gray, self.ROI_SIZE, interpolation=cv2.INTER_AREA)
        lap = laplacian_level(gray.astype(np.float32) / 255.0, self.PYR_LEVEL)

        self.buf.append((t, lap.ravel()))
        cutoff = t - self.WINDOW_SEC
        while self.buf and self.buf[0][0] < cutoff:
            self.buf.popleft()

    def fill(self):
        if len(self.buf) < 2:
            return 0.0
        span = self.buf[-1][0] - self.buf[0][0]
        return min(span / self.WINDOW_SEC, 1.0)

    def estimate(self):
        """호출 시점에만 계산. self.rr_bpm / self.rr_conf 갱신."""
        if self.fill() < self.MIN_FILL:
            self.rr_conf = 0.0
            return

        ts = np.fromiter((b[0] for b in self.buf), dtype=np.float64,
                         count=len(self.buf))
        X = np.stack([b[1] for b in self.buf])            # (N, P)

        # 실제 타임스탬프 기반 균일 격자 리샘플링 (가변 fps 대응)
        n = int((ts[-1] - ts[0]) * self.TARGET_FS)
        if n < 40:
            self.rr_conf = 0.0
            return
        grid = np.linspace(ts[0], ts[-1], n)
        Xr = np.empty((n, X.shape[1]), dtype=np.float64)
        for p in range(X.shape[1]):
            Xr[:, p] = np.interp(grid, ts, X[:, p])

        # 화소별 시간축 밴드패스 (EVM temporal filter)
        Xf = filtfilt(self.b, self.a, Xr, axis=0)
        Xf -= Xf.mean(axis=0)

        # 공통 진동 성분 = 제1주성분. 부호가 살아있어 정류가 불필요하다.
        u, s, _ = np.linalg.svd(Xf, full_matrices=False)
        sig = u[:, 0] * s[0]

        bpm, sqi = spectral_peak(sig, self.TARGET_FS,
                                 self.FREQ_LOW, self.FREQ_HIGH)
        self.rr_bpm = bpm
        # 창이 덜 찼으면 신뢰도를 비례 감쇠
        self.rr_conf = sqi * min(self.fill() / 0.9, 1.0)


# ══════════════════════════════════════════════════════
#  rPPG 심박수
# ══════════════════════════════════════════════════════

class HeartRateTracker:
    """
    open-rppg HR + 품질 지표.

    open-rppg 가 신뢰도를 제공하지 않으므로, 연속 추정치의 산포로 대체한다.
    주의: 실제로 HR 이 급변하는 순간도 산포가 커져 신뢰도가 떨어진다.
          이 경우를 놓치지 않도록 vital_anomaly 의 절대 안전범위 레이어는
          신뢰도 게이팅을 우회한다.
    """

    HR_MIN, HR_MAX = 40.0, 200.0
    SPREAD_MAX = 25.0        # 최근 창의 최대-최소가 이 값이면 신뢰도 0

    def __init__(self, model, window=4, hr_window_sec=10):
        self.model = model
        self.hr_window_sec = hr_window_sec
        self.recent = deque(maxlen=window)
        self.hr_bpm = 0.0
        self.hr_conf = 0.0

    def update(self, face_visible):
        res = self.model.hr(start=-self.hr_window_sec)
        hr = float(res["hr"]) if res and res.get("hr") else 0.0

        if not face_visible or not (self.HR_MIN <= hr <= self.HR_MAX):
            self.hr_conf = 0.0
            self.recent.clear()
            if hr > 0:
                self.hr_bpm = hr
            return

        self.hr_bpm = hr
        self.recent.append(hr)
        if len(self.recent) < self.recent.maxlen:
            self.hr_conf = 0.5           # 판단 보류
            return

        spread = max(self.recent) - min(self.recent)
        self.hr_conf = float(np.clip(1.0 - spread / self.SPREAD_MAX, 0.0, 1.0))


# ══════════════════════════════════════════════════════
#  ROI 선택
# ══════════════════════════════════════════════════════

class RoiTracker:
    """
    open-rppg 얼굴 박스에서 EVM ROI 를 유도.

    호흡은 흉곽 움직임이 얼굴 미세 움직임보다 SNR 이 훨씬 높으므로 흉곽을 우선하고,
    프레임을 벗어나면 얼굴로 폴백한다. 박스 떨림은 EVM 신호에 직접 잡음으로
    들어가므로 EMA 로 평활한다.
    """

    ALPHA = 0.25             # EMA 계수 (작을수록 강한 평활)

    def __init__(self):
        self.box = None      # 평활된 (x1, y1, x2, y2)
        self.kind = None     # "chest" | "face"

    def update(self, face_box, frame_shape):
        """face_box = (x1, y1, x2, y2) 또는 None. 반환: (roi, kind, changed)"""
        if face_box is None:
            return self.box, self.kind, False

        h, w = frame_shape[:2]
        fx1, fy1, fx2, fy2 = face_box
        fw, fh = fx2 - fx1, fy2 - fy1
        cx = (fx1 + fx2) / 2.0

        # 흉곽 후보: 얼굴 아래, 폭 2배
        c = (cx - fw, fy2 + 0.30 * fh, cx + fw, fy2 + 1.30 * fh)
        if 0 <= c[0] and 0 <= c[1] and c[2] <= w and c[3] <= h:
            target, kind = c, "chest"
        else:
            target = (fx1 + 0.15 * fw, fy1 + 0.05 * fh,
                      fx2 - 0.15 * fw, fy1 + 0.65 * fh)
            kind = "face"

        changed = kind != self.kind
        if changed or self.box is None:
            self.box, self.kind = target, kind
        else:
            a = self.ALPHA
            self.box = tuple(a * t + (1 - a) * p
                             for t, p in zip(target, self.box))
        return self.box, self.kind, changed

    def crop(self, frame):
        if self.box is None:
            return None
        x1, y1, x2, y2 = (int(round(v)) for v in self.box)
        h, w = frame.shape[:2]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 - x1 < 16 or y2 - y1 < 16:
            return None
        return frame[y1:y2, x1:x2]


# ══════════════════════════════════════════════════════
#  HUD
# ══════════════════════════════════════════════════════

def draw_hud(frame, hr, rr, fps, anomaly, roi_kind):
    """hr = (bpm, conf), rr = (bpm, conf).  cv2.putText 는 한글 미지원 → ASCII."""
    h, w = frame.shape[:2]

    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (330, 225), (10, 10, 10), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    def block(label, bpm, conf, y, hue):
        col = hue if conf > 0.3 else (130, 130, 130)
        cv2.putText(frame, label, (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1)
        txt = f"{bpm:.0f} BPM" if conf > 0.3 else "warming up..."
        cv2.putText(frame, txt, (10, y + 34),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.25, col, 2)
        cv2.rectangle(frame, (10, y + 44), (100, y + 52), (50, 50, 50), -1)
        cv2.rectangle(frame, (10, y + 44),
                      (10 + int(min(conf, 1.0) * 90), y + 52), col, -1)

    block("HEART RATE  (rPPG)", hr[0], hr[1], 26, (80, 220, 80))
    block(f"RESP RATE   (EVM/{roi_kind or '-'})", rr[0], rr[1], 108,
          (80, 180, 255))

    if anomaly is not None:
        if anomaly["critical"]:
            col, txt = (40, 40, 255), f"CRITICAL: {anomaly['critical']}"
        elif anomaly["state"] == "signal_lost":
            col, txt = (0, 165, 255), "SIGNAL LOST"
        elif anomaly["alert"]:
            col, txt = (60, 60, 255), anomaly["reason"] or "ANOMALY"
        elif anomaly["baseline"] is None:
            col = (150, 150, 150)
            txt = f"BASELINE {anomaly['progress'] * 100:.0f}%"
        else:
            col, txt = (80, 220, 80), "NORMAL"
        cv2.putText(frame, txt, (10, 212),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.48, col, 1)

    cv2.putText(frame, f"FPS: {fps:.0f}", (10, h - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (160, 160, 160), 1)
    cv2.putText(frame, "HR:60-100  RR:12-20", (w - 185, h - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (120, 120, 120), 1)
    return frame


# ══════════════════════════════════════════════════════
#  메인 루프
# ══════════════════════════════════════════════════════

def run(camera_id=0, calib_sec=180.0, min_conf=0.40):
    print("[INFO] open-rppg 모델 로딩 중...")
    model = rppg.Model("ME-flow.rlap")
    print("[INFO] 모델 로딩 완료")

    hr_tracker = HeartRateTracker(model)
    resp = RespiratorySignal()
    roi = RoiTracker()
    detector = VitalAnomalyDetector(calib_sec=calib_sec, min_conf=min_conf)
    anomaly = None

    UPDATE_INTERVAL = 2.0
    last_update = time.time()
    fps_timer, frame_cnt, measured_fps = time.time(), 0, 0.0

    print("=" * 56)
    print(f"  실시간 심박수 + 호흡수 측정 시작 (기준 학습 {calib_sec:.0f}초)")
    print("  Q / ESC : 종료")
    print("=" * 56)

    try:
        with model.video_capture(camera_id):
            for frame_rgb, box in model.preview:
                now = time.time()
                frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

                frame_cnt += 1
                if now - fps_timer >= 2.0:
                    measured_fps = frame_cnt / (now - fps_timer)
                    frame_cnt, fps_timer = 0, now

                # ── ROI 갱신 ──────────────────────────
                face_box = None
                if box is not None:
                    (y1, y2), (x1, x2) = box[0], box[1]
                    face_box = (x1, y1, x2, y2)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (80, 220, 80), 2)

                _, kind, changed = roi.update(face_box, frame.shape)
                if changed:
                    resp.clear()

                patch = roi.crop(frame)
                if patch is not None:
                    resp.push(patch, now)
                    bx = tuple(int(round(v)) for v in roi.box)
                    cv2.rectangle(frame, bx[:2], bx[2:], (0, 220, 220), 1)

                # ── 2초마다 갱신 ──────────────────────
                if now - last_update >= UPDATE_INTERVAL:
                    hr_tracker.update(face_visible=box is not None)
                    resp.estimate()

                    anomaly = detector.push(hr_tracker.hr_bpm,
                                            hr_tracker.hr_conf,
                                            resp.rr_bpm, resp.rr_conf, now=now)

                    fill = resp.fill()
                    if fill < resp.MIN_FILL:
                        rr_s = (f"창 충전중 {fill * 100:.0f}%"
                                f" (>{resp.MIN_FILL * 100:.0f}% 부터 측정)")
                    else:
                        rr_s = (f"{resp.rr_bpm:5.1f} ({resp.rr_conf:.2f})"
                                f"  fill={fill * 100:.0f}%")
                    print(f"[HR] {hr_tracker.hr_bpm:6.1f} ({hr_tracker.hr_conf:.2f})"
                          f"  [RR] {rr_s}  ROI={kind}")

                    if anomaly["critical"]:
                        print(f"  ** 절대범위 이탈: {anomaly['critical']}")
                    elif anomaly["state"] == "signal_lost":
                        print("  ** 신호 소실")
                    elif anomaly["alert"]:
                        print(f"  ** 이상징후: {anomaly['reason']} "
                              f"(score={anomaly['score']:.3f})")
                    elif anomaly["baseline"] is None:
                        print(f"  기준 학습 {anomaly['progress'] * 100:.0f}%  "
                              f"{detector.stats()}")

                    last_update = now

                frame = draw_hud(frame,
                                 (hr_tracker.hr_bpm, hr_tracker.hr_conf),
                                 (resp.rr_bpm, resp.rr_conf),
                                 measured_fps, anomaly, kind)
                cv2.imshow("Vital Monitor  (HR: rPPG | RR: EVM)", frame)

                if (cv2.waitKey(1) & 0xFF) in (ord("q"), 27):
                    print("[INFO] 종료")
                    break
    finally:
        cv2.destroyAllWindows()


if __name__ == "__main__":
    p = argparse.ArgumentParser(description="심박수 + 호흡수 실시간 측정")
    p.add_argument("--camera", type=int, default=0, help="카메라 ID")
    p.add_argument("--calib", type=float, default=180.0, help="기준 학습 초")
    p.add_argument("--min-conf", type=float, default=0.40,
                   help="SQI 게이팅 임계 (0.40: 잡음통과 1.7%%)")
    args, _ = p.parse_known_args()
    run(camera_id=args.camera, calib_sec=args.calib, min_conf=args.min_conf)

[INFO] open-rppg 모델 로딩 중...
[INFO] 모델 로딩 완료
  실시간 심박수 + 호흡수 측정 시작 (기준 학습 180초)
  Q / ESC : 종료
[HR]    0.0 (0.00)  [RR]   0.0 (0.00)  ROI=face  fill=1%
  기준 학습 0%  {'hr_conf': 1, 'rr_conf': 0, 'hr_range': 0, 'rr_range': 0, 'accepted': 0}
[HR]   65.1 (0.50)  [RR]   0.0 (0.00)  ROI=face  fill=5%
  기준 학습 0%  {'hr_conf': 1, 'rr_conf': 1, 'hr_range': 0, 'rr_range': 0, 'accepted': 0}
[HR]   90.5 (0.50)  [RR]   0.0 (0.00)  ROI=face  fill=8%
  기준 학습 0%  {'hr_conf': 1, 'rr_conf': 2, 'hr_range': 0, 'rr_range': 0, 'accepted': 0}
[HR]   91.4 (0.50)  [RR]   0.0 (0.00)  ROI=face  fill=12%
  기준 학습 0%  {'hr_conf': 1, 'rr_conf': 3, 'hr_range': 0, 'rr_range': 0, 'accepted': 0}
[HR]   90.6 (0.00)  [RR]   0.0 (0.00)  ROI=face  fill=15%
  기준 학습 0%  {'hr_conf': 2, 'rr_conf': 3, 'hr_range': 0, 'rr_range': 0, 'accepted': 0}
[HR]   90.8 (0.96)  [RR]   0.0 (0.00)  ROI=face  fill=18%
  기준 학습 0%  {'hr_conf': 2, 'rr_conf': 4, 'hr_range': 0, 'rr_range': 0, 'accepted': 0}
[HR]   93.9 (0.87)  [RR]   0.0 (0.00)  ROI=fa

Exception in thread Thread-11:
Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\envs\ai\lib\threading.py", line 980, in _bootstrap_inner
    self.run()
  File "c:\ProgramData\anaconda3\envs\ai\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\ProgramData\anaconda3\envs\ai\lib\threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "c:\ProgramData\anaconda3\envs\ai\lib\site-packages\rppg\main.py", line 723, in <lambda>
    self.run = threading.Thread(target=lambda:self.__process_video_capture(vid_path, api))
  File "c:\ProgramData\anaconda3\envs\ai\lib\site-packages\rppg\main.py", line 841, in __process_video_capture
    self.update_frame(img, ts)
  File "c:\ProgramData\anaconda3\envs\ai\lib\site-packages\rppg\main.py", line 535, in __exit__
    self.wait_completion()
  File "c:\ProgramData\anaconda3\envs\ai\lib\site-packages\rppg\main.py", line 737, in wait_completion
    self.ru

[INFO] 종료


In [1]:
"""
실시간 심박수 + 호흡수 동시 측정
  - 심박수 : open-rppg (rPPG 기반)
  - 호흡수 : EVM Motion 모드 (얼굴 미세 움직임 기반)

설치:
    pip install open-rppg
    pip install opencv-python numpy scipy
"""

import cv2
import numpy as np
import time
import argparse
from collections import deque
from scipy.signal import butter, lfilter
import rppg

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        
# ══════════════════════════════════════════════════════
#  EVM 호흡수 파이프라인
# ══════════════════════════════════════════════════════

def build_laplacian_pyramid(frame, levels=4):
    # 1. 가우시안 피라미드 먼저 만들기 (흐리게 + 축소)
    g_pyr = [frame]               # Level 0 = 원본
    for _ in range(levels):
        frame = cv2.pyrDown(frame) # 흐리게 만들고 절반 크기로 축소
        g_pyr.append(frame)

    # 2. 라플라시안 = 원본 - 흐린버전 (엣지/움직임만 남음)
    l_pyr = []
    for i in range(levels, 0, -1):
        expanded = cv2.pyrUp(g_pyr[i])         # 작은 것을 다시 확대
        laplacian = g_pyr[i-1] - expanded      # 빼면 움직임 성분만 남음
        l_pyr.append(laplacian)

    return l_pyr  # pyr[0] = 가장 섬세한 움직임 성분


def butter_bandpass(low, high, fps, order=1):
    nyq = fps / 2.0
    lo = max(0.001, min(low / nyq, 0.999))
    hi = max(0.001, min(high / nyq, 0.999))
    if lo >= hi:
        hi = min(lo + 0.05, 0.999)
    b, a = butter(order, [lo, hi], btype='band')
    return b, a


def temporal_filter(buffer, b, a):
    """버퍼(deque) 전체에 버터워스 필터 적용 후 마지막 값 반환"""
    arr = np.stack(list(buffer), axis=0).astype(np.float32)  # (N, H, W, C)
    filtered = np.zeros_like(arr)
    for c in range(arr.shape[3]):
        for h in range(arr.shape[1]):
            filtered[:, h, :, c] = lfilter(b, a, arr[:, h, :, c], axis=0)
    return filtered[-1]


def compute_bpm(signal_buffer, fps, freq_low, freq_high):
    """
    1D 시계열 신호 → FFT → 주파수 피크 → BPM 반환
    반환: (bpm, confidence 0~1)
    """
    sig = np.array(signal_buffer)
    if len(sig) < 16:
        return 0.0, 0.0

    sig = sig - np.mean(sig)
    sig *= np.hanning(len(sig))

    fft_vals = np.abs(np.fft.rfft(sig, n=len(sig) * 4))
    freqs    = np.fft.rfftfreq(len(sig) * 4, d=1.0 / fps)

    mask = (freqs >= freq_low) & (freqs <= freq_high)
    if not np.any(mask):
        return 0.0, 0.0

    band_vals  = fft_vals[mask]
    band_freqs = freqs[mask]
    peak_idx   = np.argmax(band_vals)
    peak_freq  = band_freqs[peak_idx]

    confidence = float(band_vals[peak_idx]) / (np.sum(fft_vals) + 1e-9)
    confidence = min(confidence * 10.0, 1.0)

    return peak_freq * 60.0, confidence


class RespiratoryEVM:
    """
    EVM Motion 모드 기반 실시간 호흡수 측정기

    Parameters
    ----------
    fps         : 카메라 FPS
    freq_low    : 호흡 주파수 하한 (Hz), 기본 0.15 = 9 bpm
    freq_high   : 호흡 주파수 상한 (Hz), 기본 0.60 = 36 bpm
    levels      : 라플라시안 피라미드 레벨
    buf_size    : EVM 프레임 버퍼 크기
    sig_size    : FFT용 신호 버퍼 크기
    fixed_size  : ROI 리사이즈 고정 크기 (shape 통일용)
    """

    FREQ_LOW  = 0.15   # Hz (9 bpm)
    FREQ_HIGH = 0.60   # Hz (36 bpm)

    def __init__(self, fps=30.0, levels=4, buf_size=256,
                 sig_size=512, fixed_size=(64, 64)):
        self.fps        = fps
        self.levels     = levels
        self.fixed_size = fixed_size

        self.frame_buf = deque(maxlen=buf_size)
        self.sig_buf   = deque(maxlen=sig_size)

        self.b, self.a = butter_bandpass(
            self.FREQ_LOW, self.FREQ_HIGH, fps
        )

        # 결과
        self.rr_bpm  = 0.0
        self.rr_conf = 0.0

    def push(self, roi_bgr):
        """
        BGR ROI 프레임을 받아 신호 추출.
        충분히 쌓이면 self.rr_bpm / self.rr_conf 갱신.
        """
        # 고정 크기 리사이즈 (shape 불일치 방지)
        roi = cv2.resize(roi_bgr, self.fixed_size,
                         interpolation=cv2.INTER_LINEAR)
        roi_f = roi.astype(np.float32) / 255.0

        # 라플라시안 피라미드 첫 번째 레벨 (엣지 성분)
        pyr    = build_laplacian_pyramid(roi_f, self.levels)
        target = pyr[0]

        self.frame_buf.append(target)

        min_frames = self.frame_buf.maxlen // 2
        if len(self.frame_buf) < min_frames:
            return

        # 템포럴 필터링 → 증폭 → 신호 추출
        filtered = temporal_filter(self.frame_buf, self.b, self.a)
        self.sig_buf.append(float(np.mean(np.abs(filtered))))

    def update_bpm(self):
        """sig_buf가 충분히 차면 BPM 재계산"""
        if len(self.sig_buf) < 32:
            return
        self.rr_bpm, self.rr_conf = compute_bpm(
            self.sig_buf, self.fps, self.FREQ_LOW, self.FREQ_HIGH
        )
        # 생리적 범위 클리핑
        if self.rr_bpm < 6 or self.rr_bpm > 40:
            self.rr_conf *= 0.3   # 범위 벗어나면 신뢰도 낮춤

    def ready(self):
        return len(self.frame_buf) >= self.frame_buf.maxlen // 2


# ══════════════════════════════════════════════════════
#  HUD 그리기 유틸
# ══════════════════════════════════════════════════════

def draw_hud(frame, hr_bpm, hr_conf, rr_bpm, rr_conf,
             fps, hr_ready, rr_ready):
    h, w = frame.shape[:2]

    # 반투명 배경 패널
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (310, 185), (10, 10, 10), -1)
    cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)

    # 색상: 신뢰도 낮으면 회색, 높으면 색상
    hr_color = (80,  220, 80)  if (hr_conf > 0.3 and hr_ready) else (130, 130, 130)
    rr_color = (80,  180, 255) if (rr_conf > 0.3 and rr_ready) else (130, 130, 130)

    # ── 심박수 ────────────────────────────────────────
    cv2.putText(frame, "HEART RATE  (rPPG)",
                (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.52, hr_color, 1)
    hr_txt = f"{hr_bpm:.0f} BPM" if hr_ready else "warming up..."
    cv2.putText(frame, hr_txt,
                (10, 62), cv2.FONT_HERSHEY_SIMPLEX, 1.3, hr_color, 2)
    # 신뢰도 바
    bar_w = int(min(hr_conf, 1.0) * 90)
    cv2.rectangle(frame, (10, 72), (100, 80), (50, 50, 50), -1)
    cv2.rectangle(frame, (10, 72), (10 + bar_w, 80), hr_color, -1)

    # ── 호흡수 ────────────────────────────────────────
    cv2.putText(frame, "RESP RATE   (EVM)",
                (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.52, rr_color, 1)
    rr_txt = f"{rr_bpm:.0f} BPM" if rr_ready else "warming up..."
    cv2.putText(frame, rr_txt,
                (10, 145), cv2.FONT_HERSHEY_SIMPLEX, 1.3, rr_color, 2)
    # 신뢰도 바
    bar_w = int(min(rr_conf, 1.0) * 90)
    cv2.rectangle(frame, (10, 155), (100, 163), (50, 50, 50), -1)
    cv2.rectangle(frame, (10, 155), (10 + bar_w, 163), rr_color, -1)

    # ── FPS ───────────────────────────────────────────
    cv2.putText(frame, f"FPS: {fps:.0f}",
                (10, h - 10), cv2.FONT_HERSHEY_SIMPLEX,
                0.45, (160, 160, 160), 1)

    # ── 정상 범위 안내 ────────────────────────────────
    cv2.putText(frame, "HR:60-100  RR:12-20",
                (w - 185, h - 10), cv2.FONT_HERSHEY_SIMPLEX,
                0.42, (120, 120, 120), 1)

    return frame


# ══════════════════════════════════════════════════════
#  메인 루프
# ══════════════════════════════════════════════════════

def run(camera_id=0):
    """
    open-rppg 심박수 + EVM 호흡수 동시 측정 메인 루프

    종료: Q 또는 ESC
    """

    # ── open-rppg 모델 초기화 ─────────────────────────
    print("[INFO] open-rppg 모델 로딩 중...")
    model = rppg.Model('ME-flow.rlap')
    print("[INFO] 모델 로딩 완료")

    # ── FPS 측정용 ────────────────────────────────────
    fps_timer   = time.time()
    frame_cnt   = 0
    measured_fps = 30.0

    # ── 호흡수 EVM ────────────────────────────────────
    rr_evm = RespiratoryEVM(fps=measured_fps)

    # ── BPM 갱신 타이머 ───────────────────────────────
    last_bpm_time = time.time()
    BPM_INTERVAL  = 2.0   # 초

    # ── 결과 변수 ─────────────────────────────────────
    hr_bpm, hr_conf = 0.0, 0.0
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    )
    last_face = None

    print("=" * 50)
    print("  실시간 심박수 + 호흡수 측정 시작")
    print("  Q / ESC : 종료")
    print("=" * 50)

    # ── open-rppg 비디오 캡처 컨텍스트 진입 ─────────
    with model.video_capture(camera_id):
        for frame_rgb, box in model.preview:
            # open-rppg는 RGB 반환 → BGR로 변환
            frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

            # ── FPS 측정 ──────────────────────────────
            frame_cnt += 1
            elapsed = time.time() - fps_timer
            if elapsed >= 2.0:
                measured_fps = frame_cnt / elapsed
                frame_cnt    = 0
                fps_timer    = time.time()

            # ── 얼굴 감지 (EVM ROI용) ─────────────────
            gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(
                gray, scaleFactor=1.1,
                minNeighbors=5, minSize=(80, 80)
            )
            if len(faces) > 0:
                last_face = max(faces, key=lambda f: f[2] * f[3])

            # ── open-rppg ROI 박스 그리기 ─────────────
            if box is not None:
                y1, y2 = box[0]
                x1, x2 = box[1]
                cv2.rectangle(frame, (x1, y1), (x2, y2),
                              (80, 220, 80), 2)

            # ── EVM ROI 추출 + 호흡 신호 push ─────────
            if last_face is not None:
                fx, fy, fw, fh = last_face
                # 이마~코 영역
                ry1 = fy + int(fh * 0.05)
                ry2 = fy + int(fh * 0.65)
                rx1 = fx + int(fw * 0.15)
                rx2 = fx + int(fw * 0.85)

                roi = frame[ry1:ry2, rx1:rx2]
                if roi.size > 0:
                    rr_evm.push(roi)

                # EVM ROI 박스 표시 (노란색)
                cv2.rectangle(frame, (rx1, ry1), (rx2, ry2),
                              (0, 220, 220), 1)

            # ── BPM 갱신 (2초마다) ────────────────────
            now = time.time()
            if now - last_bpm_time >= BPM_INTERVAL:

                # 심박수: open-rppg
                result = model.hr(start=-10)
                if result and result.get('hr'):
                    hr_bpm  = float(result['hr'])
                    hr_conf = 0.9   # open-rppg는 자체 신뢰도 미제공
                    # 범위 벗어나면 신뢰도 낮춤
                    if hr_bpm < 40 or hr_bpm > 200:
                        hr_conf = 0.1

                # 호흡수: EVM
                rr_evm.update_bpm()

                # 콘솔 출력
                hr_str = f"{hr_bpm:.1f} BPM" if hr_bpm > 0 else "측정 중..."
                rr_str = f"{rr_evm.rr_bpm:.1f} BPM" if rr_evm.rr_bpm > 0 else "측정 중..."
                print(f"[심박수] {hr_str:12s}  |  [호흡수] {rr_str}")

                if box is not None:
                    y1, y2 = box[0]
                    x1, x2 = box[1]
                    print(f"[rPPG 박스] 좌상({x1},{y1}) 우상({x2},{y1}) "
                          f"우하({x2},{y2}) 좌하({x1},{y2})")

                last_bpm_time = now

            # ── HUD 오버레이 ──────────────────────────
            frame = draw_hud(
                frame,
                hr_bpm=hr_bpm,
                hr_conf=hr_conf,
                rr_bpm=rr_evm.rr_bpm,
                rr_conf=rr_evm.rr_conf,
                fps=measured_fps,
                hr_ready=(hr_bpm > 0),
                rr_ready=rr_evm.ready(),
            )

            cv2.imshow("Vital Monitor  (HR: rPPG | RR: EVM)", frame)

            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), 27):
                print("[INFO] 종료")
                break

    cv2.destroyAllWindows()


# ══════════════════════════════════════════════════════
#  진입점
# ══════════════════════════════════════════════════════

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="심박수 + 호흡수 실시간 측정")
    parser.add_argument("--camera", type=int, default=0,
                        help="카메라 ID (기본: 0)")
    args, _ = parser.parse_known_args()
    run(camera_id=args.camera)

[INFO] open-rppg 모델 로딩 중...
[INFO] 모델 로딩 완료
  실시간 심박수 + 호흡수 측정 시작
  Q / ESC : 종료
[심박수] 측정 중...       |  [호흡수] 측정 중...
[rPPG 박스] 좌상(289,207) 우상(416,207) 우하(416,363) 좌하(289,363)
[심박수] 106.9 BPM     |  [호흡수] 측정 중...
[rPPG 박스] 좌상(289,207) 우상(416,207) 우하(416,363) 좌하(289,363)
[심박수] 67.9 BPM      |  [호흡수] 측정 중...
[rPPG 박스] 좌상(289,207) 우상(416,207) 우하(416,363) 좌하(289,363)
[심박수] 68.0 BPM      |  [호흡수] 35.3 BPM
[rPPG 박스] 좌상(289,207) 우상(416,207) 우하(416,363) 좌하(289,363)
[심박수] 67.7 BPM      |  [호흡수] 33.6 BPM
[rPPG 박스] 좌상(289,207) 우상(416,207) 우하(416,363) 좌하(289,363)
[심박수] 67.8 BPM      |  [호흡수] 32.1 BPM
[rPPG 박스] 좌상(249,209) 우상(394,209) 우하(394,386) 좌하(249,386)
[심박수] 66.9 BPM      |  [호흡수] 9.9 BPM
[rPPG 박스] 좌상(212,193) 우상(402,193) 우하(402,426) 좌하(212,426)
[심박수] 65.9 BPM      |  [호흡수] 9.4 BPM
[rPPG 박스] 좌상(248,185) 우상(428,185) 우하(428,405) 좌하(248,405)
[심박수] 69.8 BPM      |  [호흡수] 9.0 BPM
[rPPG 박스] 좌상(227,205) 우상(403,205) 우하(403,421) 좌하(227,421)
[심박수] 73.9 BPM      |  [호흡수] 24.1 BPM
[rPPG 박스] 좌상(212,142) 우

Exception in thread Thread-5:
Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\envs\ai\lib\threading.py", line 980, in _bootstrap_inner
    self.run()
  File "c:\ProgramData\anaconda3\envs\ai\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\ProgramData\anaconda3\envs\ai\lib\threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "c:\ProgramData\anaconda3\envs\ai\lib\site-packages\rppg\main.py", line 723, in <lambda>
    self.run = threading.Thread(target=lambda:self.__process_video_capture(vid_path, api))
  File "c:\ProgramData\anaconda3\envs\ai\lib\site-packages\rppg\main.py", line 841, in __process_video_capture
    self.update_frame(img, ts)
  File "c:\ProgramData\anaconda3\envs\ai\lib\site-packages\rppg\main.py", line 535, in __exit__
    self.wait_completion()
  File "c:\ProgramData\anaconda3\envs\ai\lib\site-packages\rppg\main.py", line 737, in wait_completion
    self.run

[INFO] 종료
